---

### Formalização do Modelo Estocástico

O problema da trajetória acadêmica e do fluxo institucional no curso de Ciência da Computação da UFRJ é modelado por meio de Cadeias de Markov de Tempo Discreto (DTMC) acopladas a uma Simulação de Monte Carlo com reamostragem via Bootstrap.

#### 1. Espaço de Estados ($S$)

O tempo do modelo avança em saltos discretos anuais ($t$). O espaço de estados $S$ é definido por três possíveis condições de vínculo do discente com a instituição:

* $A$: **Ativo** (Estado Transiente)
* $E$: **Evadido** (Estado Absorvente)
* $F$: **Formado** (Estado Absorvente)

#### 2. Matriz de Transição Estocástica ($P$)

$$P = \begin{bmatrix}
pA_A & pE_A & pF_A \\
0 & 1 & 0 \\
0 & 0 & 1
\end{bmatrix}$$

$$pA_A = 1 - (pE_A + pF_A)$$

#### 3. Simulação 1: Análise de Coorte (Sobrevivência e Decaimento)

Projeta a vida acadêmica de uma única turma de $N$ ingressantes por $t=10$ anos. Sistema **fechado** (sem novos ingressos após $t=0$), Cadeia **Não-Homogênea no Tempo**.

* **Vetor de Estado Inicial:** $V_0 = [N, 0, 0]$
* **Fase 1 (Ciclo Básico, $t < 4$):** $pF_A = 0$ (trava curricular). Urna: evasão de calouros.
* **Fase 2 (Ciclo Final, $t \ge 4$):** Urnas de evasão tardia e formatura ativadas.

#### 4. Simulação 2: Fluxo Institucional e Vazão

Estima o *steady-state* do curso como sistema integral no longo prazo. Sistema **aberto** e **Homogêneo no Tempo**.

$$V_{t+1} = V_t \times P_{macro} + [I, 0, 0]$$

onde $I$ é a entrada anual de calouros. O estado inicial ($t=0$) usa o último censo disponível (2024).

#### 5. Modelagem de Risco e Incerteza (Bootstrap)

$pE$ e $pF$ não são tratadas como determinísticas: a cada iteração da Simulação de Monte Carlo, sorteia-se (com reposição) um valor real observado nos microdados do INEP (2015-2024), excluindo os anos de 2020 e 2021 (Período Letivo Excepcional).

In [ ]:
# -------------------------------------------------------------------------
# Setup: pacotes e dados
# -------------------------------------------------------------------------
if (!require("jsonlite")) install.packages("jsonlite")
library(jsonlite)

# Urnas de bootstrap (geradas por bootsrap.py) -- caminho local, sem
# dependência de link externo do Google Drive
urnas_json   <- fromJSON("matrizes/urnas_bootstrap.json")
urnas_coorte <- urnas_json$simulacao_1_coorte
urnas_macro  <- urnas_json$simulacao_2_fluxo

# Estado inicial da Simulação 2: último censo disponível (2024)
censo <- read.csv("dados/DADOS_CURSOS_REF_TOTAL.csv")
linha_censo_2024 <- subset(censo, CO_CURSO == 85783 & NU_ANO_CENSO == 2024)
A0_censo <- as.integer(linha_censo_2024$QT_MAT[1])
ING_2024 <- as.integer(linha_censo_2024$QT_ING[1])

cat(sprintf("Estado do ultimo censo (2024): Ativos matriculados = %d | Ingressantes do ano = %d\n",
            A0_censo, ING_2024))

## Simulação 1 — Recorte de Coorte

In [ ]:
# -------------------------------------------------------------------------
# Simulação 1: Cadeia de Markov Não-Homogênea (Coorte fechada)
# -------------------------------------------------------------------------
set.seed(123)
N_simulacoes <- 3000
N_alunos_inicial <- 60
t_max <- 10
estados <- c("Ativo", "Evadido", "Formado")

historico <- array(0, dim = c(N_simulacoes, t_max, 3))
dimnames(historico)[[3]] <- estados

for (s in 1:N_simulacoes) {
  status <- rep("Ativo", N_alunos_inicial)

  for (t in 1:t_max) {
    # Regra do relogio curricular: trava de formatura para t < 4
    if (t < 4) {
      pE_atual <- sample(urnas_coorte$evasao_inicial_pE_A, 1)
      pF_atual <- 0
    } else {
      pE_atual <- sample(urnas_coorte$evasao_tardia_pE_R, 1)
      pF_atual <- sample(urnas_coorte$formatura_tardia_pF_R, 1)
    }
    pA_atual <- 1 - (pE_atual + pF_atual)

    # Roleta vetorizada apenas para quem ainda esta Ativo
    ativos_idx <- which(status == "Ativo")
    if (length(ativos_idx) > 0) {
      status[ativos_idx] <- sample(estados, length(ativos_idx), replace = TRUE,
                                     prob = c(pA_atual, pE_atual, pF_atual))
    }
    historico[s, t, ] <- c(sum(status == "Ativo"), sum(status == "Evadido"), sum(status == "Formado"))
  }
}

# Snapshot final (Ano 10) usado nas analises de distribuicao
resultados <- historico[, t_max, ]

In [ ]:
cat("--- Simulacao 1: Coorte de Ciencia da Computacao UFRJ ---\n")
cat("Mediana de alunos apos 10 anos:\n")
print(apply(resultados, 2, median))

cat("\nIntervalos de Confianca 95% (2.5% - 97.5%):\n")
print(apply(resultados, 2, quantile, probs = c(0.025, 0.975)))

limites_inf <- apply(resultados, 2, quantile, probs = 0.05)
limites_sup <- apply(resultados, 2, quantile, probs = 0.95)
cat(sprintf(
  "\nCom 90%% de confianca, apos 10 anos, a coorte estara dentro do intervalo:\nAtivo: [%d a %d] | Evadidos: [%d a %d] | Formados: [%d a %d]\n",
  limites_inf["Ativo"], limites_sup["Ativo"],
  limites_inf["Evadido"], limites_sup["Evadido"],
  limites_inf["Formado"], limites_sup["Formado"]
))

In [ ]:
par(mfrow = c(1, 2))
for (estado in estados) {
  dados <- resultados[, estado]

  hist(dados, prob = TRUE, main = paste("Histograma:", estado),
       xlab = "N de Alunos", col = "lightblue", border = "white")
  curve(dnorm(x, mean = mean(dados), sd = sd(dados)), add = TRUE, col = "darkblue", lwd = 2)

  plot(ecdf(dados), main = paste("ECDF:", estado),
       xlab = "N de Alunos", ylab = "Probabilidade Acumulada")
  q90 <- quantile(dados, 0.9)
  abline(v = q90, col = "red", lwd = 2, lty = 2)

  cat(sprintf("\n--- %s --- Percentil 0.9: %.1f\n", estado, q90))
}
par(mfrow = c(1, 1))

In [ ]:
cores <- c("Ativo" = "blue", "Evadido" = "red", "Formado" = "green")

plot(1:t_max, colMeans(historico[, , "Ativo"]), type = "n", ylim = c(0, N_alunos_inicial),
     xlab = "Ano", ylab = "Quantidade de Alunos", main = "Evolucao Temporal da Coorte")

for (est in estados) {
  medias <- colMeans(historico[, , est])
  lim_inf <- apply(historico[, , est], 2, quantile, probs = 0.1)
  lim_sup <- apply(historico[, , est], 2, quantile, probs = 0.9)

  polygon(c(1:t_max, rev(1:t_max)), c(lim_inf, rev(lim_sup)),
          col = adjustcolor(cores[est], alpha.f = 0.2), border = NA)
  lines(1:t_max, medias, col = cores[est], lwd = 3)
}
legend("right", legend = estados, col = cores, lwd = 3)

## Simulação 2 — Fluxo Institucional

In [ ]:
# -------------------------------------------------------------------------
# Simulacao 2: Cadeia Aberta e Homogenea (Fluxo institucional)
# -------------------------------------------------------------------------
set.seed(123)
N_simulacoes <- 3000
t_max_fluxo <- 10
estados_fluxo <- c("Ativo", "Evadido", "Formado")

A0 <- A0_censo                    # QT_MAT em 2024 (ultimo censo)
N_ingressantes_anual <- ING_2024  # QT_ING em 2024 (nao usar valores de anos anteriores)

# historico_fluxo guarda, por ano: estoque de Ativos ao final do ano e o
# FLUXO (nao acumulado) de evasao/formatura ocorrido naquele ano
historico_fluxo <- array(0, dim = c(N_simulacoes, t_max_fluxo, 3))
dimnames(historico_fluxo)[[3]] <- estados_fluxo

for (s in 1:N_simulacoes) {
  ativos <- A0

  for (t in 1:t_max_fluxo) {
    pE <- sample(urnas_macro$evasao_macro_pE, 1)
    pF <- sample(urnas_macro$formatura_macro_pF, 1)
    pA <- 1 - (pE + pF)
    if (pA < 0) {                       # salvaguarda (nao ocorre com os dados atuais)
      soma <- pE + pF; pE <- pE / soma; pF <- pF / soma; pA <- 0
    }

    # V_(t+1) = V_t x P_macro + [I, 0, 0]:
    # a transicao e aplicada sobre o estoque ANTES da entrada de calouros,
    # porque foi assim que pE/pF foram calibrados (MAT_ANTERIOR no bootsrap.py)
    transicoes <- rmultinom(1, size = ativos, prob = c(pA, pE, pF))
    evadidos_ano <- as.integer(transicoes[2, 1])
    formados_ano <- as.integer(transicoes[3, 1])
    ativos <- as.integer(transicoes[1, 1]) + N_ingressantes_anual

    historico_fluxo[s, t, ] <- c(ativos, evadidos_ano, formados_ano)
  }
}

In [ ]:
final <- historico_fluxo[, t_max_fluxo, ]
medianas <- apply(final, 2, median)
ic_contagens <- apply(final, 2, quantile, probs = c(0.025, 0.975))

taxa_evasao_media <- mean(urnas_macro$evasao_macro_pE)
taxa_formatura_media <- mean(urnas_macro$formatura_macro_pF)

# Ponto de equilibrio teorico (steady-state): A* = I / (pE + pF)
steady_state <- N_ingressantes_anual / (taxa_evasao_media + taxa_formatura_media)

cat("--- Simulacao 2: Fluxo Institucional (Ciencia da Computacao) ---\n")
cat(sprintf("Estado inicial (censo 2024): Ativos = %d | Ingressantes/ano = %d\n", A0, N_ingressantes_anual))
cat(sprintf("\nMediana de Ativos em 2034: %.0f  (tendencia: %+.0f em relacao a 2024)\n",
            medianas["Ativo"], medianas["Ativo"] - A0))
cat(sprintf("IC95%% de Ativos em 2034: [%.0f, %.0f]\n", ic_contagens[1, "Ativo"], ic_contagens[2, "Ativo"]))
cat(sprintf("\nEvasao no ano 10 (fluxo, nao acumulado): mediana = %.0f  IC95%% = [%.0f, %.0f]\n",
            medianas["Evadido"], ic_contagens[1, "Evadido"], ic_contagens[2, "Evadido"]))
cat(sprintf("Formatura no ano 10 (fluxo, nao acumulado): mediana = %.0f  IC95%% = [%.0f, %.0f]\n",
            medianas["Formado"], ic_contagens[1, "Formado"], ic_contagens[2, "Formado"]))
cat(sprintf("\nPonto de equilibrio (steady-state) estimado: %.0f alunos ativos\n", steady_state))
cat(sprintf("Como A0 = %d esta %s do equilibrio, a tendencia esperada e de %s no tamanho do curso.\n",
            A0, ifelse(A0 > steady_state, "acima", "abaixo"),
            ifelse(A0 > steady_state, "QUEDA", "CRESCIMENTO")))

In [ ]:
t_range <- 1:t_max_fluxo
mediana_ativos  <- apply(historico_fluxo[, , "Ativo"], 2, median)
q10_ativos <- apply(historico_fluxo[, , "Ativo"], 2, quantile, probs = 0.1)
q90_ativos <- apply(historico_fluxo[, , "Ativo"], 2, quantile, probs = 0.9)

mediana_evadido <- apply(historico_fluxo[, , "Evadido"], 2, median)
mediana_formado <- apply(historico_fluxo[, , "Formado"], 2, median)

plot(t_range, mediana_ativos, type = "n", ylim = c(0, max(q90_ativos) * 1.1),
     xlab = "Ano (a partir de 2024)", ylab = "Quantidade de Alunos",
     main = "Evolucao do Tamanho do Curso (Ativos)")
polygon(c(t_range, rev(t_range)), c(q10_ativos, rev(q90_ativos)),
        col = adjustcolor("blue", alpha.f = 0.2), border = NA)
lines(t_range, mediana_ativos, col = "blue", lwd = 3)
abline(h = A0, col = "gray40", lty = 2)
abline(h = steady_state, col = "darkorange", lty = 2)
legend("topright", legend = c("Mediana Ativos", "A0 (2024)", "Steady-state"),
       col = c("blue", "gray40", "darkorange"), lty = c(1, 2, 2), lwd = c(3, 1, 1))

plot(t_range, mediana_evadido, type = "l", col = "red", lwd = 3,
     ylim = c(0, max(mediana_evadido, mediana_formado) * 1.3),
     xlab = "Ano (a partir de 2024)", ylab = "N de alunos no ano",
     main = "Fluxo Anual de Evasao e Formatura (nao acumulado)")
lines(t_range, mediana_formado, col = "green", lwd = 3)
legend("topleft", legend = c("Evasao no ano", "Formatura no ano"), col = c("red", "green"), lwd = 3)